# Embedding & Indexing — Amazon Product Dataset 2020 (Home & Kitchen)

Covers task backlog §1.3 "Embedding & Indexing", building on
[1.2 process data.ipynb](1.2%20process%20data.ipynb)'s output (`products_clean.csv`):

- filter to the `Home & Kitchen` demo slice decided in §1.1
- build embedding text per product (`title` + `features`, with fallback)
- embed locally with `sentence-transformers` (free, no API key — see project decision log)
- persist to a Chroma collection with a metadata payload for filtering/citations
- sanity-check the index with real similarity queries

**Config-driven by design** (per the assignment's model-agnostic requirement, applied here
for consistency even though embeddings aren't the "LLM" the requirement targets): every
provider/model/path below comes from `.env` (see [.env.example](../../.env.example) at the
repo root) with sensible defaults, so swapping the embedding provider later is a config
change, not a rewrite.

Self-contained, pre-executed. CSV in for input; Chroma's own on-disk format for the index
(a vector store isn't representable as CSV — everything upstream of it still is).

In [1]:
# !pip install pandas sentence-transformers chromadb python-dotenv

## Setup & config

In [2]:
import os

import pandas as pd
from dotenv import load_dotenv

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
load_dotenv(os.path.join(REPO_ROOT, ".env"))  # no-op if .env doesn't exist

INPUT_CSV = os.path.join(REPO_ROOT, "Code", "data", "processed", "products_clean.csv")

EMBEDDING_PROVIDER = os.environ.get("EMBEDDING_PROVIDER", "sentence-transformers")
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "all-MiniLM-L6-v2")
CHROMA_PERSIST_DIR = os.path.join(REPO_ROOT, os.environ.get(
    "CHROMA_PERSIST_DIR", "Code/data/processed/chroma_db"
))
CHROMA_COLLECTION = os.environ.get("CHROMA_COLLECTION", "products_home_kitchen")
DEMO_CATEGORY = "Home & Kitchen"

print(f"EMBEDDING_PROVIDER = {EMBEDDING_PROVIDER}")
print(f"EMBEDDING_MODEL    = {EMBEDDING_MODEL}")
print(f"CHROMA_PERSIST_DIR = {CHROMA_PERSIST_DIR}")
print(f"CHROMA_COLLECTION  = {CHROMA_COLLECTION}")

if EMBEDDING_PROVIDER != "sentence-transformers":
    raise NotImplementedError(
        f"EMBEDDING_PROVIDER={EMBEDDING_PROVIDER!r} is configured but this notebook only "
        "implements the sentence-transformers path so far. Add the branch for this "
        "provider here before switching .env to it."
    )

EMBEDDING_PROVIDER = sentence-transformers
EMBEDDING_MODEL    = all-MiniLM-L6-v2
CHROMA_PERSIST_DIR = /Users/pareeyaporn/Desktop/ADSP 32028 Applied Generative AI - Agents and Multimodal Intelligence/Project/Code/data/processed/chroma_db
CHROMA_COLLECTION  = products_home_kitchen


## Load cleaned data & filter to the demo slice

From [1.2 process data.ipynb](1.2%20process%20data.ipynb)'s output — already deduped, brand-
and price-normalized. Filtering to `Home & Kitchen` here (not upstream in preprocessing) is
what makes expanding the index later just a matter of loosening this filter and re-running,
per the earlier scope discussion.

In [3]:
df_all = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df_all):,} cleaned products (all categories).")

df = df_all[df_all["category_top"] == DEMO_CATEGORY].copy().reset_index(drop=True)
print(f"Filtered to category_top == {DEMO_CATEGORY!r}: {len(df):,} products.")
df.head(3)

Loaded 9,930 cleaned products (all categories).
Filtered to category_top == 'Home & Kitchen': 706 products.


,id,title,brand,category_top,category_full,price,price_per_unit,price_unit,features,technical_details,product_specification,url,image_url
0,cc2083338a16c3fe2f7895289d2e98fe,"ARTSCAPE Etched Glass 24"" x 36"" Window Film, 2...",ARTSCAPE Etched,Home & Kitchen,Home & Kitchen | Home Décor | Window Treatment...,12.99,NaN,NaN,Make sure this fits by entering your model num...,show up to 2 reviews by default Product Descri...,ProductDimensions:72x36x0inches|ItemWeight:11....,https://www.amazon.com/ARTSCAPE-Etched-Glass-W...,https://images-na.ssl-images-amazon.com/images...
1,39f1b8a2129315da0288cd058b6b6086,Flash Furniture 25''W x 45''L Trapezoid Red HP...,Flash Furniture,Home & Kitchen,Home & Kitchen | Furniture | Kids' Furniture |...,117.26,NaN,NaN,Collaborative Trapezoid Activity Table | Table...,Go to your orders and start the return Select ...,ASIN:B00777W7GE|ShippingWeight:39pounds(Viewsh...,https://www.amazon.com/Flash-Furniture-Trapezo...,https://images-na.ssl-images-amazon.com/images...
2,a11d9462309527143094a0f68bce0a58,Industro Stainless Steel Hose Clamps,Industro Stainless,Home & Kitchen,Home & Kitchen | Home Décor | Kids' Room Décor,34.27,NaN,NaN,Make sure this fits by entering your model num...,Go to your orders and start the return Select ...,ProductDimensions:2x0.6x2inches|ItemWeight:1.3...,https://www.amazon.com/Industro-Stainless-Stee...,https://images-na.ssl-images-amazon.com/images...


## Build embedding text

`title` + `features` is the primary source (per the task backlog). Where `features` is
missing, fall back to `technical_details` then `product_specification` rather than
embedding the title alone.

In [4]:
def build_embedding_text(row):
    parts = [row["title"]]
    secondary = row["features"]
    if pd.isna(secondary):
        secondary = row["technical_details"]
    if pd.isna(secondary):
        secondary = row["product_specification"]
    if pd.notna(secondary):
        parts.append(str(secondary))
    return " ".join(parts).strip()


df["embedding_text"] = df.apply(build_embedding_text, axis=1)

before = len(df)
df = df[df["embedding_text"].str.len() > 0].reset_index(drop=True)
print(f"Rows with usable embedding text: {len(df):,} / {before:,}")
print()
print("Sample embedding text:")
print(" ", df["embedding_text"].iloc[0][:160])

Rows with usable embedding text: 706 / 706

Sample embedding text:
  ARTSCAPE Etched Glass 24" x 36" Window Film, 24-by-36-Inch Make sure this fits by entering your model number. | The visual effect of textured glass and stained 


## Load the embedding model

Local, free, no API key (`sentence-transformers` / `all-MiniLM-L6-v2`, 384 dimensions).
First run downloads the model weights (~80MB) and caches them under `~/.cache`; later runs
load from cache.

In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Loaded {EMBEDDING_MODEL}, embedding dimension = {model.get_sentence_embedding_dimension()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded all-MiniLM-L6-v2, embedding dimension = 384


/var/folders/dt/hz0jqyys0fz2ksjdx5kkxm800000gn/T/ipykernel_4585/2735288613.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Loaded {EMBEDDING_MODEL}, embedding dimension = {model.get_sentence_embedding_dimension()}")


## Compute embeddings

In [6]:
embeddings = model.encode(
    df["embedding_text"].tolist(), show_progress_bar=True, batch_size=64
)
print(f"Embeddings shape: {embeddings.shape}")

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Embeddings shape: (706, 384)


## Build the metadata payload

Chroma metadata values must be `str`/`int`/`float`/`bool` — no `NaN`/`None` — so this drops
missing fields per row rather than passing them through. `id` doubles as the citation
`doc_id` back to the source row.

In [7]:
METADATA_FIELDS = ["id", "title", "brand", "category_top", "price",
                    "price_per_unit", "price_unit", "url"]


def build_metadata(row):
    meta = {}
    for field in METADATA_FIELDS:
        val = row[field]
        if pd.isna(val):
            continue
        if isinstance(val, (int, float)):
            val = float(val)
        else:
            val = str(val)
        meta[field] = val
    return meta


metadatas = [build_metadata(row) for _, row in df.iterrows()]
ids = df["id"].astype(str).tolist()
documents = df["embedding_text"].tolist()

print("Sample metadata record:")
print(" ", metadatas[0])

Sample metadata record:
  {'id': 'cc2083338a16c3fe2f7895289d2e98fe', 'title': 'ARTSCAPE Etched Glass 24" x 36" Window Film, 24-by-36-Inch', 'brand': 'ARTSCAPE Etched', 'category_top': 'Home & Kitchen', 'price': 12.99, 'url': 'https://www.amazon.com/ARTSCAPE-Etched-Glass-Window-Film/dp/B000Q3PRYA'}


## Build/persist the Chroma collection

Idempotent: drops and recreates the collection each run, so re-running this notebook after
a data change never leaves stale/duplicate vectors behind.

In [8]:
import chromadb

os.makedirs(CHROMA_PERSIST_DIR, exist_ok=True)
client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)

try:
    client.delete_collection(CHROMA_COLLECTION)
except Exception:
    pass  # collection didn't exist yet

collection = client.create_collection(
    name=CHROMA_COLLECTION,
    metadata={"embedding_model": EMBEDDING_MODEL, "category": DEMO_CATEGORY},
)

BATCH_SIZE = 200
for start in range(0, len(ids), BATCH_SIZE):
    end = start + BATCH_SIZE
    collection.add(
        ids=ids[start:end],
        embeddings=embeddings[start:end].tolist(),
        metadatas=metadatas[start:end],
        documents=documents[start:end],
    )

print(f"Indexed {collection.count():,} products into collection {CHROMA_COLLECTION!r}")
print(f"Persisted to: {os.path.relpath(CHROMA_PERSIST_DIR, REPO_ROOT)}")

Indexed 706 products into collection 'products_home_kitchen'
Persisted to: Code/data/processed/chroma_db


## Sanity check — query the index

**Important finding from the first pass of this check:** the queries originally tried here ("eco-friendly kitchen cleaner", "non-stick frying pan", "coffee maker under $50") returned poor matches — cake pans, coffee mugs, a toaster. Investigating why (see [detailed_task_backlog.md](../../Project%20Overview/detailed_task_backlog.md) §1.3) found this is a **data coverage problem, not a retrieval-quality problem**: searching the *entire* 9,930-row cleaned dataset for "frying pan"/"skillet" returns **zero** matches, and "coffee maker" returns exactly **one**. Even the `Kitchen & Dining` sub-category (55 rows, any top-level category) turns out to be dominated by kids' lunch boxes and novelty lunch kits, not cookware.

This `Home & Kitchen` slice is actually **kids'-goods-heavy**: of 706 rows, the real sub-category breakdown is Home Décor (198), Furniture (195, mostly Kids' Furniture), Bedding (177, mostly Kids' Bedding), Event & Party Supplies (72), and Kitchen & Dining (53, mostly lunch boxes) — so the queries below were changed to match what the data actually contains, per the project decision to reframe the demo pitch rather than chase a different dataset. Read the results below for real — this is the check that caught the mismatch in the first place.


In [9]:
sample_queries = [
    "insulated lunch box for kids",
    "kids bedding set",
    "toy storage organizer",
    "birthday party decorations",
]

for query in sample_queries:
    query_embedding = model.encode([query])[0].tolist()
    results = collection.query(query_embeddings=[query_embedding], n_results=3)
    print(f"Query: {query!r}")
    for title, meta, dist in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        price = meta.get("price")
        price_str = f"${price:.2f}" if price is not None else "?"
        print(f"    [{dist:.3f}] {meta.get('title', title)[:70]}  ({price_str})")
    print()

Query: 'insulated lunch box for kids'
    [0.589] Wildkin Kids Insulated Lunch Box for Boys and Girls, Perfect Size for   ($12.77)
    [0.649] Wildkin Kids Insulated Lunch Box for Boys and Girls, Perfect Size for   ($10.88)
    [0.687] Wildkin 33600 Monsters Lunch Box, Insulated, Moisture Resistant, and E  ($14.99)

Query: 'kids bedding set'
    [0.730] Heritage Kids Color Splash Bedding Set, Queen, Pink  ($99.99)
    [0.759] Mi-Zone Kids Dinosaur Dreams Twin Kids Bedding Sets for Boys - Grey, B  ($54.88)
    [0.787] Urban Habitat Kids Finn Twin/Twin Xl Bedding Sets Boys Quilt Set - Gre  ($39.99)



Query: 'toy storage organizer'
    [0.538] SONGMICS UGKR48WT Children’s Toy Storage Unit White  ($48.99)
    [0.672] Fantasy Fields - Sunny Safari Toy Organizer with Rolling Storage Box,   ($103.99)
    [0.702] Phoenix Home Espresso Collection Kid's Toy Storage Organizer with 12 P  ($49.00)

Query: 'birthday party decorations'
    [0.869] #40 Celebration Candle | White/Red | Party Supply  ($4.75)
    [0.885] Happy Anniversary Gold - Large Foil Letter Banner  ($5.28)
    [0.908] Metallic Happy Birthday Banner (pink w/silver gltrd pink ltrs) Party A  ($8.00)



## Summary

- **Input:** [products_clean.csv](processed/products_clean.csv), filtered to
  `category_top == "Home & Kitchen"`.
- **Embedding:** `sentence-transformers` / `all-MiniLM-L6-v2`, local, free, 384 dimensions —
  config-driven via `EMBEDDING_PROVIDER`/`EMBEDDING_MODEL` in `.env` (see
  [.env.example](../../.env.example)).
- **Vector store:** Chroma, persisted to `Code/data/processed/chroma_db/`, collection name
  from `CHROMA_COLLECTION`. Rebuild is idempotent — just re-run this notebook.
- **Metadata payload** per vector: `id, title, brand, category_top, price, price_per_unit,
  price_unit, url` — `id` is the citation `doc_id`.
- **Expanding later:** loosen or remove the `category_top == "Home & Kitchen"` filter above
  and re-run; `products_clean.csv` already has the full dataset, nothing upstream needs
  redoing (see the earlier scope discussion in the task backlog).
- **Demo pitch reframed to match the data:** this slice is kids'-goods-heavy (bedding,
  furniture, décor, party supplies, lunch boxes), not cookware/cleaning-heavy — the
  assignment brief's "eco-friendly stainless-steel cleaner" example isn't supported by
  this dataset in any category slice (verified against the full 9,930-row cleaned data,
  not just this one). Project pitch/example queries were updated accordingly rather than
  chasing a different dataset — see the root README and task backlog.
- Sanity-checked above with real queries — read those results, not this summary, to judge
  actual retrieval quality.